## Silver — `populacao_estimada` (População por Município, IBGE via Base dos Dados)

**Origem:** `workspace.bronze.estimativa_populacional` → **Destino:** `workspace.silver.populacao_estimada`

- **Grão:** 1 linha por município x ano (`codigo_municipio` + `ano`), deduplicado.
- **Transformações:**
  - Correção de tipos com conversão tolerante: `ano` inteiro e `populacao` numérico positivo — valor malformado vira null e é contabilizado como inválido.
  - Filtros de qualidade (cada regra contabilizada no relatório):
    - `codigo_municipio` não nulo, com 7 dígitos e **com registro em** `workspace.silver.municipios`;
    - `ano` entre **1991 e 2025**;
    - `populacao` não nula e > 0;
    - `sigla_uf` presente e igual à sigla oficial do município (conciliação com a base IBGE/Dim).
- **Relatório de qualidade:** quantidade de inválidos por regra + % sobre o total bronze.
- **Linhagem:** Base dos Dados CSV → `bronze.estimativa_populacional` → limpeza/validação/conciliação com `silver.municipios` → `silver.populacao_estimada`.

In [0]:
%run ../shared/_setup

In [0]:
from pyspark.sql import functions as F
from data_pipeline import (
    save_table,
    add_column_comments,
    add_table_comment,
    resumo_invalidos,
    condicao_valida,
    para_double_seguro,
    nulo_se_vazio,
)
from catalogo.populacao import (
    SILVER_POPULACAO_COMMENTS,
    SILVER_POPULACAO_TABLE_COMMENT,
)

In [0]:
SOURCE_TABLE = "workspace.bronze.estimativa_populacional"
TARGET_TABLE = "workspace.silver.populacao_estimada"
MUNICIPIOS_TABLE = "workspace.silver.municipios"

# Cobertura temporal da tabela br_ibge_populacao.municipio (Base dos Dados)
ANO_MIN = 1991
ANO_MAX = 2025

# Contrato de saída silver.populacao_estimada
COLUNAS_FINAIS = [
    "codigo_municipio",
    "sigla_uf",
    "ano",
    "populacao",
]

In [0]:
df_bronze = spark.table(SOURCE_TABLE)
total_bronze = df_bronze.count()
print(f"Bronze: {total_bronze:,} linhas | colunas: {df_bronze.columns}")

# Conversão tolerante (ANSI-safe): valor malformado vira null e é contabilizado no relatório
df = df_bronze.select(
    F.trim(F.col("id_municipio").cast("string")).alias("codigo_municipio"),
    nulo_se_vazio(F.col("sigla_uf")).alias("sigla_uf"),
    F.when(F.trim(F.col("ano").cast("string")).rlike("^[0-9]{4}$"), F.trim(F.col("ano").cast("string")).cast("int")).alias("ano"),
    para_double_seguro(F.col("populacao")).cast("long").alias("populacao"),
)
display(df.limit(5))

In [0]:
# Left join com a base oficial de municípios (IBGE): sem correspondência => inválido;
# traz também a sigla oficial para conciliar com a sigla informada no arquivo origem
df_municipios = (
    spark.table(MUNICIPIOS_TABLE)
    .select(
        "codigo_municipio",
        F.col("sigla_uf").alias("sigla_uf_oficial"),
    )
    .distinct()
    .withColumn("municipio_valido", F.lit(True))
)

df = df.join(df_municipios, on="codigo_municipio", how="left")
com_registro = df.filter(F.col("municipio_valido")).count()
pct_registro = round(100 * com_registro / total_bronze, 2) if total_bronze else 0.0
print(f"Códigos com registro em municipios: {com_registro:,} ({pct_registro}%)")

In [0]:
REGRAS_INVALIDOS = {
    "codigo_nulo_ou_nao_numerico": (
        F.col("codigo_municipio").isNull()
        | ~F.col("codigo_municipio").rlike("^[0-9]{7}$")
    ),
    "codigo_sem_registro_em_municipios": ~F.coalesce(F.col("municipio_valido"), F.lit(False)),
    "ano_fora_do_intervalo_1991_2025": F.col("ano").isNull() | ~F.col("ano").between(ANO_MIN, ANO_MAX),
    "populacao_nula_ou_nao_positiva": F.col("populacao").isNull() | (F.col("populacao") <= 0),
    "sigla_uf_divergente_da_oficial": (
        F.col("sigla_uf").isNull()
        | F.col("sigla_uf_oficial").isNull()
        | (F.col("sigla_uf") != F.col("sigla_uf_oficial"))
    ),
}

In [0]:
df_relatorio = resumo_invalidos(spark, df, REGRAS_INVALIDOS)
print(f"Total bronze avaliado: {total_bronze:,}")
display(df_relatorio)

In [0]:
# Mantém apenas registros válidos em todas as regras
df = df.filter(condicao_valida(REGRAS_INVALIDOS))

# Deduplicação pela PK natural (município x ano)
antes_dedup = df.count()
df = df.dropDuplicates(["codigo_municipio", "ano"])
print(f"Válidos: {antes_dedup:,} | Após dedup por codigo_municipio+ano: {df.count():,}")

df = df.select(*COLUNAS_FINAIS)
display(df.limit(10))

In [0]:
save_table(df, TARGET_TABLE)
add_column_comments(
    spark,
    TARGET_TABLE,
    SILVER_POPULACAO_COMMENTS
)
add_table_comment(spark, TARGET_TABLE, SILVER_POPULACAO_TABLE_COMMENT)
print(f"Tabela {TARGET_TABLE} persistida: {spark.table(TARGET_TABLE).count():,} linhas")

In [0]:
total = spark.table(TARGET_TABLE).count()
pk_distintos = spark.table(TARGET_TABLE).select("codigo_municipio", "ano").distinct().count()
print(f"Total: {total:,} | PK distintos (município x ano): {pk_distintos:,} | Duplicatas PK: {total - pk_distintos:,}")
assert total == pk_distintos, "Quebra de unicidade PK em silver.populacao_estimada"
display(spark.sql(f"SELECT ano, count(*) AS qtd_municipios, sum(populacao) AS populacao_total FROM {TARGET_TABLE} GROUP BY ano ORDER BY ano"))
display(spark.sql(f"DESCRIBE TABLE {TARGET_TABLE}"))